# Memory

In [1]:
!pip install langchain_community

In [2]:
#os_key Setting

from dotenv import load_dotenv
import os
# .env 파일 로드(api key load) 
#GOOGLE_API_KEY 
#LANGCHAIN_API_KEY 
#TAVILY_API_KEY 
#HUGGINGFACEHUB_API_TOKEN 
#COHERE_API_KEY 

load_dotenv()

True

In [ ]:
#open ai case

In [2]:
from langchain_openai import ChatOpenAI

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
#gemini case

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
model = ChatGoogleGenerativeAI(model="gemini-1.5-pro")

In [6]:
from langchain_core.messages import HumanMessage

In [7]:
model.invoke([HumanMessage(content="안녕 내 이름은 Liam이야")]).content

'안녕하세요, Liam! 만나서 반갑습니다. 무엇을 도와드릴까요?\n'

In [8]:
model.invoke([HumanMessage(content="내 이름이 뭐야?")]).content

'저는 대규모 언어 모델이기 때문에 당신의 이름을 알 수 없습니다. 저는 당신과의 대화에서 당신이 제공한 정보에만 접근할 수 있습니다.  당신의 이름을 알려주시겠어요?\n'

In [ ]:
#model 자체에는 memory 기능이 없기 때문에 이전 대화 내용을 전달해야 대답할 수 있다.

In [9]:
from langchain_core.messages import AIMessage

In [10]:
model.invoke(
    [
        HumanMessage(content="안녕 내 이름은 Liam이야"),
        AIMessage(content="반가워 Liam! 만나서 반가워요. 무엇을 도와드릴까요?"),
        HumanMessage(content="내 이름이 뭐야?"),
    ]
).content

'Liam이라고 하셨습니다. 😊\n'

## Message History

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [12]:
#store라는 dict을 만든다.
store = {}

#session_id(str)을 입력 받아서 ChatMessageHistory를 반환한다.
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:                    #store에 dict에 입력 받은 'session_id' 키가 없으면
        store[session_id] = ChatMessageHistory()   #store[session_id]를 생성하고 값은 ChatMessageHistory()로 입력한다.
    return store[session_id]                       #store[sesion_id]에 해당하는 ChatMessageHistory()를 반환한다.

#llm model과 get_session_history를 runnable로 묶어 준다.
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [13]:
#configurable 키값으로 dict을 생성한다. session_id
config = {"configurable": {"session_id": "session_1"}}

In [14]:
response = with_message_history.invoke(
    [HumanMessage(content="안녕 내 이름은 Liam이야")],
    config=config,
)

response.content

'안녕하세요, Liam! 반갑습니다. 무엇을 도와드릴까요?\n'

In [15]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐야?")],
    config=config,
)

response.content

'Liam입니다.  방금 말씀해주셨어요. 😊\n'

In [16]:
config = {"configurable": {"session_id": "session_2"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐야?")],
    config=config,
)

response.content

'저는 대화형 AI이기 때문에 당신의 이름을 알 수 없습니다. 당신의 이름을 알려주시면 기억해 둘게요!\n'

In [17]:
config = {"configurable": {"session_id": "session_1"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐야?")],
    config=config,
)

response.content

'Liam입니다. 😊\n'

## Prompt templates 과 함께 쓰기

### Chain 만들기

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [19]:
#'messages'라는 dict의 key 값들로 입력된 값들이 MessagesPlaceholder에 쌓인다.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 도움이 되는 AI Assistant이다. 모든 질문에 최선을 다해 답변하라.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [20]:
response = chain.invoke({"messages": [HumanMessage(content="안녕 난 Liam이야")]})

response.content

'안녕하세요, Liam! 무엇을 도와드릴까요?'

In [21]:
response2 = chain.invoke({"messages": [HumanMessage(content="나는 아이스크림을 좋아해")]})

response2.content

'저도 아이스크림을 좋아해요! 어떤 맛을 가장 좋아하세요?'

In [22]:
response3 = chain.invoke({"messages": [HumanMessage(content="내이름과 내가 좋아하는 음식을 알려줘줘")]})

response3.content

'죄송하지만 해당 정보는 저에게 제공되지 않았습니다. 이름과 좋아하는 음식을 알려주시면 기억해 두었다가 다음에 요청하실 때 알려드릴 수 있습니다.'

In [23]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [26]:
config3 = {"configurable": {"session_id": "session_3"}}

In [27]:
response = with_message_history.invoke(
    [HumanMessage(content="안녕 난 David야")],
    config=config3,
)

response.content

'안녕하세요, David님. 무엇을 도와드릴까요?'

In [28]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐야?")],
    config=config3,
)

response.content

'제가 알기로는 당신의 이름은 David입니다. 방금 말씀해 주셨죠. 😊\n'

In [29]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 도움이 되는 AI Assistant이다. 다음의 언어로 대답하라: {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [30]:
response = chain.invoke(
    {"messages": [HumanMessage(content="안녕 나는 밥이야")], "language": "Spanish"}
)

response.content

'Hola Bob, ¿en qué te puedo ayudar hoy?\n'

In [32]:
response = chain.invoke(
    {"messages": [HumanMessage(content="안녕 나는 밥이야")], "language": "English"}
)

response.content

'안녕, 밥! 무엇을 도와드릴까요?\n'

### RunnableWithMessageHistory 로 감싸기

In [33]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [34]:
config = {"configurable": {"session_id": "session_4"}}

In [35]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="안녕 나는 Liam이야")], "language": "Japanese"},
    config=config,
)

response.content

'こんにちは、Liamさん。\n\nはじめまして。何かお手伝いできることはありますか？\n'

In [36]:
model.invoke(f'"{response.content}"가 한국어로 뭐야?').content

'한국어로는 다음과 같습니다:\n\n안녕하세요, Liam 씨.\n\n처음 뵙겠습니다. 무엇을 도와드릴까요?\n\n\n좀 더 자연스럽게는 아래와 같이 표현할 수도 있습니다.\n\n* 안녕하세요, Liam님. 처음 뵙겠습니다. 뭐 도와드릴까요?  (Liam 씨 보다 Liam님이 조금 더 부드러운 표현)\n* 안녕하세요 Liam씨.  무슨 일이시죠? (처음 뵙겠습니다를 생략하고 간략하게)\n* 안녕하세요 Liam님.  도와드릴 일 있으신가요? (처음 뵙겠습니다를 생략하고 간략하게)\n\n어떤 표현을 사용할지는 상황과 맥락에 따라 선택하면 됩니다.\n'

## Conversation History 관리하기

### Chain 만들기

In [55]:
from langchain_core.runnables import RunnablePassthrough


def filter_messages(messages, k=10):
    msgs = messages[-k:]
    print(f"len msgs : {len(msgs)}")
    [print(f"{'ai' if type(msg) == AIMessage else 'human'}: {msg.content}")for msg in msgs]
    return msgs


# x["messages"] : 11개의 message list(messages 10ea +HumanMessage 1ea)
# filter_messages(x["messages"]) : 11개의 list중 마지막 10개만 반환
# 마지막 10개의 messages list만 전달달

chain = (
    RunnablePassthrough.assign(messages=lambda x: filter_messages(x["messages"])) 
    | prompt
    | model
)

In [56]:
messages = [
    HumanMessage(content="안녕하세요! 저는 밥이에요"),
    AIMessage(content="안녕하세요!"),
    HumanMessage(content="저는 바닐라 아이스크림을 좋아해요"),
    AIMessage(content="좋네요"),
    HumanMessage(content="2 + 2가 뭐죠?"),
    AIMessage(content="4에요"),
    HumanMessage(content="고마워요"),
    AIMessage(content="문제 없어요!"),
    HumanMessage(content="재미있어요?"),
    AIMessage(content="네, 재미있어요!"),
]

In [57]:
#message

response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="내이름이 뭐야?")],
        "language": "English",
    }
)
response.content  # 기억하지 못함

len msgs : 10
ai: 안녕하세요!
human: 저는 바닐라 아이스크림을 좋아해요
ai: 좋네요
human: 2 + 2가 뭐죠?
ai: 4에요
human: 고마워요
ai: 문제 없어요!
human: 재미있어요?
ai: 네, 재미있어요!
human: 내이름이 뭐야?


'저는 당신의 이름을 모르겠어요. 저는 당신의 메시지에 접근할 수 없어요.\n\n'

In [58]:
#기존 10개의 message에 umanMessage(content="내이름이 뭐야?")를 더해서 총 'messages'에는 11개의 message가 입력된다.
len(messages + [HumanMessage(content="내이름이 뭐야?")])

11

In [59]:
#기존 10개의 message에 umanMessage(content="내이름이 뭐야?")를 더해서 총 'messages'에는 11개의 message가 입력된다.
messages + [HumanMessage(content="내이름이 뭐야?")]

[HumanMessage(content='안녕하세요! 저는 밥이에요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='저는 바닐라 아이스크림을 좋아해요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='좋네요', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='2 + 2가 뭐죠?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4에요', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='고마워요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문제 없어요!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='재미있어요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네, 재미있어요!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내이름이 뭐야?', additional_kwargs={}, response_metadata={})]

In [60]:
# lambda x: filter_messages(x["messages"])의 의미
# 입력된 dict{}의 'messages'값 : 총 11개의 message list
k = {
        "messages": messages + [HumanMessage(content="내이름이 뭐야?")],
        "language": "English",
    }
k['messages']

[HumanMessage(content='안녕하세요! 저는 밥이에요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='저는 바닐라 아이스크림을 좋아해요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='좋네요', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='2 + 2가 뭐죠?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4에요', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='고마워요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문제 없어요!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='재미있어요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네, 재미있어요!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내이름이 뭐야?', additional_kwargs={}, response_metadata={})]

In [61]:
RunnablePassthrough.assign(messages=lambda x: filter_messages(x["messages"])) 

RunnableAssign(mapper={
  messages: RunnableLambda(lambda x: filter_messages(x['messages']))
})

In [62]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="내가 좋아하는 아이스크림은 뭐야?")],
        "language": "English",
    }
)
response.content  # 기억함

len msgs : 10
ai: 안녕하세요!
human: 저는 바닐라 아이스크림을 좋아해요
ai: 좋네요
human: 2 + 2가 뭐죠?
ai: 4에요
human: 고마워요
ai: 문제 없어요!
human: 재미있어요?
ai: 네, 재미있어요!
human: 내가 좋아하는 아이스크림은 뭐야?


'바닐라 아이스크림을 좋아한다고 하셨습니다.\n'

### RunnableWithMessageHistory 로 감싸기

In [63]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

config = {"configurable": {"session_id": "session_5"}}

In [64]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="내 이름이 뭐야?")],
        "language": "English",
    },
    config=config,
)

response.content

len msgs : 10
ai: 안녕하세요!
human: 저는 바닐라 아이스크림을 좋아해요
ai: 좋네요
human: 2 + 2가 뭐죠?
ai: 4에요
human: 고마워요
ai: 문제 없어요!
human: 재미있어요?
ai: 네, 재미있어요!
human: 내 이름이 뭐야?


'죄송하지만 사용자의 이름을 모릅니다. 저는 개인 정보를 저장하지 않으며, 이전 대화 내용도 기억하지 않습니다. 사용자의 이름이 무엇인가요?\n'

In [65]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="대화를 참고해서 내가 좋아하는 아이스크림은 뭐야?")],
        "language": "English",
    },
    config=config,
)

response.content

len msgs : 10
ai: 좋네요
human: 2 + 2가 뭐죠?
ai: 4에요
human: 고마워요
ai: 문제 없어요!
human: 재미있어요?
ai: 네, 재미있어요!
human: 내 이름이 뭐야?
ai: 죄송하지만 사용자의 이름을 모릅니다. 저는 개인 정보를 저장하지 않으며, 이전 대화 내용도 기억하지 않습니다. 사용자의 이름이 무엇인가요?

human: 대화를 참고해서 내가 좋아하는 아이스크림은 뭐야?


'저는 이전 대화 내용을 기억하지 않기 때문에 어떤 아이스크림을 좋아하는지 알 수 없습니다. 좋아하는 아이스크림을 알려주시겠어요?\n'

## Streaming

In [66]:
store.pop("session_6")

KeyError: 'session_6'

In [67]:
config = {"configurable": {"session_id": "session_6"}}
for r in with_message_history.stream(
    {
        "messages": [HumanMessage(content="안녕 재밌는 이야기하나 해줄래?")],
        "language": "Korean",
    },
    config=config,
):
    print(r.content, end="")

len msgs : 1
human: 안녕 재밌는 이야기하나 해줄래?
옛날 옛날 아주 먼 옛날, 구름 위에 떠 있는 작은 마을이 있었습니다. 이 마을 사람들은 모두 날개가 달려 자유롭게 하늘을 날아다녔죠. 그런데 유독 한 아이, 구름이만 날지 못했습니다. 구름이는 날개 대신 커다란 귀를 가지고 있었는데, 그 귀로는 세상 모든 소리를 들을 수 있었어요.

다른 아이들은 구름이를 놀리며 "너는 날지도 못하는 땅꼬마"라고 불렀습니다. 구름이는 슬펐지만, 아무 말도 하지 못했습니다.  대신 구름이는 자신의 큰 귀를 이용해 마을 사람들을 돕기 시작했습니다. 멀 곳에서 다가오는 폭풍우 소리를 듣고 미리 알려주기도 하고, 길 잃은 양의 울음소리를 듣고 찾아주기도 했죠.

어느 날, 갑자기 거대한 독수리가 마을을 습격했습니다. 날개가 있는 마을 사람들은 모두 두려움에 떨며 숨기 바빴지만, 독수리는 너무 강했습니다. 그때, 구름이는 독수리의 날갯짓 소리에서 약점을 발견했습니다. 독수리의 왼쪽 날개가 다쳐서 미세하게 다른 소리가 났던 것이죠.

구름이는 용기를 내어 큰 소리로 외쳤습니다. "독수리의 왼쪽 날개를 공격하세요!" 마을 사람들은 처음엔 주저했지만, 구름이의 확신에 찬 목소리에 힘을 얻어 독수리의 왼쪽 날개를 집중 공격했습니다.  결국 독수리는 패배하고 도망쳤습니다.

그 후로 마을 사람들은 구름이의 큰 귀가 얼마나 소중한지 깨달았습니다. 더 이상 구름이를 놀리지 않았고, 오히려 그의 능력을 존경했습니다. 구름이는 날지는 못했지만, 자신의 특별한 능력으로 마을을 구한 영웅이 되었답니다. 그리고 구름이는 자신의 큰 귀가 날개보다 더 소중한 선물이라는 것을 알게 되었습니다.
